<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 6 · Informe que responde preguntas
**Analista de datos · Power BI + Tableau** — Visuales, segmentaciones y la página del comité · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/06_informe_que_responde.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-06)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Elegir el visual según la pregunta y ver qué pasa con la elección equivocada.
2. Ver la página del comité con las cifras que debe mostrar tu Power BI.
3. Relacionar cada visual de Power BI con su equivalente en Tableau.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Definir las seis medidas (las mismas de DAX)
def medidas(v, m):
    """v: líneas de venta con el filtro de la celda; m: metas con el mismo filtro."""
    ok = v[v.estado != "Anulada"]                      # CALCULATE(..., Ventas[estado] <> "Anulada")
    venta, fact, meta = ok.subtotal.sum(), ok.id_factura.nunique(), m.meta_subtotal.sum()
    return {"Venta neta": venta, "Meta": meta,
            "Cumplimiento": venta / meta if meta else np.nan,          # DIVIDE([Venta neta], [Meta])
            "Facturas": fact,                                            # DISTINCTCOUNT(id_factura)
            "Ticket promedio": venta / fact if fact else np.nan,         # DIVIDE([Venta neta], [Facturas])
            "Margen %": ok.margen_bruto.sum() / venta if venta else np.nan}
ventas_ene = pd.read_csv(ruta("nuevo_mes/ventas_2026_01.csv"), parse_dates=["fecha"])
metas_ene = pd.read_csv(ruta("nuevo_mes/metas_2026_01.csv"))
V = (pd.concat([ventas, ventas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
     .merge(clientes[["id_cliente", "segmento"]], on="id_cliente").assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
M = pd.concat([metas, metas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
def formato(nombre, x):
    if pd.isna(x): return "(vacío)"
    return porc(x) if nombre in ("Cumplimiento", "Margen %") else miles(x)
display(Markdown("Medidas listas: Venta neta, Meta, Cumplimiento, Facturas, Ticket promedio y Margen %."))

## 1 · Pregunta y visual
Elige una pregunta del comité y un visual. El cuaderno lo dibuja con los datos reales y te dice si responde la pregunta.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Este visual responde esta pregunta? |
> | **Qué se grafica** | Depende de tu elección: tarjeta, barras, líneas, torta o matriz |
> | **Por qué esta forma** | Ver el visual equivocado con datos reales es la mejor forma de entender por qué no sirve. |
> | **Qué se descartó** | — |
> | **Cómo se lee** | Si tienes que explicar el gráfico para que se entienda la respuesta, no es el visual adecuado. |
> | **En Excel 365** | Panel Visualizaciones de Power BI: el icono de cada visual. |

In [ ]:
# @title Gráfico 1 · Prueba un visual
pregunta = "¿Qué sedes cumplieron la meta?"  # @param ["¿Cuánto vendimos en 2025?", "¿Qué sedes cumplieron la meta?", "¿Cómo fue la venta mes a mes frente a la meta?", "¿Cuánto pesa cada canal?"]
visual = "Barras"  # @param ["Tarjeta", "Barras", "Líneas", "Torta", "Matriz"]
MEJOR = {"¿Cuánto vendimos en 2025?": ["Tarjeta"], "¿Qué sedes cumplieron la meta?": ["Barras"],
         "¿Cómo fue la venta mes a mes frente a la meta?": ["Líneas"], "¿Cuánto pesa cada canal?": ["Barras", "Torta"]}
v25, m25 = V[V.anio == 2025], M[M.anio == 2025]
ok25 = v25[v25.estado != "Anulada"]
if pregunta.startswith("¿Qué sedes"):
    datos = pd.Series({c: medidas(v25[v25.ciudad == c], m25[m25.ciudad == c])["Cumplimiento"] for c in ORDEN_CIUDADES}).sort_values(); es_pct = True
elif pregunta.startswith("¿Cómo fue"):
    datos = ok25.groupby("mes")["subtotal"].sum() / 1e6; es_pct = False
elif pregunta.startswith("¿Cuánto pesa"):
    datos = ok25.groupby("canal")["subtotal"].sum(); datos = (datos / datos.sum()).sort_values(); es_pct = True
else:
    datos = pd.Series({"Venta neta 2025": ok25.subtotal.sum()}); es_pct = False
fig = go.Figure()
if visual == "Tarjeta":
    fig.add_trace(go.Indicator(mode="number", value=datos.iloc[-1] * (100 if es_pct else 1),
                               number=dict(suffix=" %" if es_pct else "", valueformat=",.1f" if es_pct else ",.0f"), title=dict(text=str(datos.index[-1]))))
elif visual == "Barras":
    fig.add_bar(x=datos.values, y=[str(i) for i in datos.index], orientation="h", marker_color=AZUL,
                text=[porc(v) if es_pct else miles(v) for v in datos.values], textposition="outside", cliponaxis=False)
    if pregunta.startswith("¿Qué sedes"): fig.add_vline(x=1, line_dash="dash", annotation_text="Meta 100 %")
elif visual == "Líneas":
    fig.add_scatter(x=[str(i) for i in datos.index], y=datos.values, mode="lines+markers", line=dict(color=AZUL, width=3), name="Venta")
    if pregunta.startswith("¿Cómo fue"):
        fig.add_scatter(x=[str(i) for i in datos.index], y=m25.groupby("mes")["meta_subtotal"].sum().values / 1e6, name="Meta", line=dict(color=GRIS, dash="dash"))
elif visual == "Torta":
    fig.add_pie(labels=[str(i) for i in datos.index], values=datos.values, hole=0.5, sort=False)
else:
    fig.add_table(header=dict(values=["", "Valor"]), cells=dict(values=[[str(i) for i in datos.index], [porc(v) if es_pct else miles(v) for v in datos.values]]))
bien = visual in MEJOR[pregunta]
fig.update_layout(title=("✓ " if bien else "✗ ") + pregunta + ("" if bien else f" · mejor: {' o '.join(MEJOR[pregunta])}"), height=380, showlegend=visual in ("Líneas", "Torta"))
fig.show()

## 2 · La página del comité

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo debe verse la página del comité en Power BI? |
> | **Qué se grafica** | Tres tarjetas, barras de cumplimiento por sede con formato condicional y la línea mensual de venta contra meta |
> | **Por qué esta forma** | Es la página del Taller P2: arriba las cifras, al centro la comparación y la evolución. |
> | **Qué se descartó** | Diez visuales: el comité tendría que buscar el mensaje. |
> | **Cómo se lee** | Cambia el canal como si usaras la segmentación: las cifras de las tarjetas deben coincidir con tu Power BI. |
> | **En Excel 365** | Tarjetas + Gráfico de barras agrupadas + Gráfico de líneas + Segmentación de datos por Ventas[canal]. |

In [ ]:
# @title Gráfico 2 · La página del comité (con segmentación de canal)
canal = "Todos"  # @param ["Todos", "Presencial", "Televentas", "E-commerce"]
from plotly.subplots import make_subplots
vf = v25 if canal == "Todos" else v25[v25.canal == canal]
tot = medidas(vf, m25)
cum_s = pd.Series({c: medidas(vf[vf.ciudad == c], m25[m25.ciudad == c])["Cumplimiento"] for c in ORDEN_CIUDADES}).sort_values()
mensual = vf[vf.estado != "Anulada"].groupby("mes")["subtotal"].sum() / 1e6
fig = make_subplots(rows=2, cols=3, specs=[[{"type": "indicator"}] * 3, [{"type": "xy", "colspan": 2}, None, {"type": "xy"}]],
                    row_heights=[0.28, 0.72], subplot_titles=["", "", "", "Cumplimiento por sede", "Venta mensual y meta (millones)"])
for i, (k, suf, vf_) in enumerate([("Venta neta", "", ",.0f"), ("Cumplimiento", " %", ".1f"), ("Ticket promedio", "", ",.0f")], start=1):
    val = tot[k] * (100 if k == "Cumplimiento" else 1)
    fig.add_trace(go.Indicator(mode="number", value=val, number=dict(suffix=suf, valueformat=vf_, font=dict(size=30)), title=dict(text=k, font=dict(size=14))), row=1, col=i)
fig.add_trace(go.Bar(x=cum_s.values, y=cum_s.index, orientation="h", marker_color=[MAGENTA if c < 1 else GRIS for c in cum_s.values],
                     text=[porc(c) for c in cum_s.values], textposition="outside", cliponaxis=False, showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=[MESES[m - 1] for m in mensual.index], y=mensual.values, mode="lines", line=dict(color=AZUL, width=3), showlegend=False), row=2, col=3)
if canal == "Todos":
    fig.add_trace(go.Scatter(x=MESES, y=m25.groupby("mes")["meta_subtotal"].sum().values / 1e6, mode="lines", line=dict(color=GRIS, dash="dash"), showlegend=False), row=2, col=3)
fig.update_xaxes(tickformat=".0%", range=[0, cum_s.max() * 1.2], row=2, col=1)
fig.update_layout(title=f"Cierre 2025 · canal: {canal}" + (f" · {cum_s.index[0]} es la sede más lejos de la meta" if canal == "Todos" else " (la meta no tiene canal: el cumplimiento compara contra la meta total)"),
                  height=560, margin=dict(t=90))
fig.show()

## 3 · El mismo informe en Tableau

| En Power BI | En Tableau Public |
|---|---|
| Tarjeta | Hoja con la medida en **Texto** |
| Gráfico de barras agrupadas | Dimensión en **Filas**, medida en **Columnas**, ordenar |
| Gráfico de líneas | Fecha (mes) en **Columnas**, medida en **Filas**, marca **Línea** |
| Segmentación de datos | Campo en **Filtros** › **Mostrar filtro** |
| Formato condicional | Campo calculado en **Color** |
| Página | **Dashboard** que junta las hojas |

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Qué visual responde «¿cómo fue la venta mes a mes frente a la meta?»?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Tarjeta", "Barras", "Líneas", "Torta"]
revisar(respuesta, "Líneas", """La evolución en el tiempo se lee como una línea, con la meta como referencia.""")

In [ ]:
# @title Reto 2 · En la página del comité con todos los canales, ¿qué sede quedó más lejos de su meta?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
revisar(respuesta, pd.Series({c: medidas(v25[v25.ciudad == c], m25[m25.ciudad == c])["Cumplimiento"] for c in ORDEN_CIUDADES}).idxmin(), """Es la primera barra magenta del gráfico 2.""")

In [ ]:
# @title Reto 3 · En Tableau, ¿cómo se llama lo que en Power BI es una página de informe?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Hoja", "Dashboard", "Historia", "Libro"]
revisar(respuesta, "Dashboard", """Cada hoja es un visual; el dashboard las junta.""")

## Apóyate en IA
Para pasar un visual de Power BI a Tableau:

```text
En Power BI hice [TIPO DE VISUAL] con [CAMPO] en el eje y [MEDIDA] en valores, filtrado por [FILTRO].
Explícame paso a paso cómo hacer lo mismo en Tableau Public: qué va en Columnas, Filas, Marcas y Filtros.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero las preguntas; después un visual por pregunta.
* Arriba las cifras, al centro la comparación y la evolución, el filtro a la vista.
* La lógica es la misma en Power BI y en Tableau: cambian los nombres.

**Siguiente:** [Laboratorio · Taller P2](https://omarli-360.github.io/powerbi-tableau-analista/#taller-2)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>